# 🎙️ Moshi Inference on RunPod — Complete Setup Guide

**Moshi** is a speech-text foundation model for real-time, full-duplex spoken dialogue.  
This notebook walks you through the **entire pipeline** on a RunPod GPU instance:

1. 🖥️ Verify CUDA & GPU environment
2. 📦 Install all dependencies
3. 🔧 Clone & install the Moshi package
4. 🚀 Run direct Python inference from a `.wav` file
5. 🌐 Launch the Moshi server & expose it via Gradio tunnel
6. 🎵 Upload / load a `.wav` file, validate format, run inference
7. 💾 Save & play the output audio in the notebook
8. 🛠️ Troubleshooting common RunPod issues

> **Hardware requirement:** A GPU with at least **24 GB VRAM** (e.g. A100-40GB, A100-80GB, RTX 3090, RTX 4090).  
> For 12 GB GPUs (e.g. RTX 3080), see the troubleshooting section.

---

## 📋 Step 1 — Verify GPU, CUDA & System Prerequisites

In [ ]:
# ── 1.1  System & GPU overview ──────────────────────────────────────────────
import subprocess, sys

print("=" * 60)
print("PYTHON VERSION:", sys.version)
print("=" * 60)

# nvidia-smi
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "nvidia-smi not found — are you on a GPU instance?")

In [ ]:
# ── 1.2  PyTorch CUDA check ─────────────────────────────────────────────────
import torch

print("PyTorch version   :", torch.__version__)
print("CUDA available    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version      :", torch.version.cuda)
    print("GPU count         :", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f"  GPU {i}           : {props.name}  |  {vram_gb:.1f} GB VRAM")
    # Quick tensor round-trip to confirm CUDA is functional
    t = torch.ones(3, 3).cuda()
    print("\nCUDA smoke test   : PASSED ✅")
else:
    print("\n⚠️  CUDA is NOT available. Moshi requires a CUDA-capable GPU.")
    print("   Ensure you launched a GPU-enabled pod in RunPod.")

In [ ]:
# ── 1.3  Disk space & RAM ───────────────────────────────────────────────────
import shutil, os

total, used, free = shutil.disk_usage("/")
print(f"Disk  total : {total // 2**30} GB")
print(f"Disk  used  : {used  // 2**30} GB")
print(f"Disk  free  : {free  // 2**30} GB")

with open("/proc/meminfo") as f:
    meminfo = {line.split()[0].rstrip(":"): line.split()[1] for line in f}
total_ram = int(meminfo["MemTotal"]) // 1024
free_ram  = int(meminfo["MemFree"])  // 1024
print(f"\nRAM   total : {total_ram} MB")
print(f"RAM   free  : {free_ram}  MB")

# Moshi model weights are ~15 GB; warn if low disk space
if free // 2**30 < 20:
    print("\n⚠️  Less than 20 GB free disk space — consider using a larger volume.")

---
## 📦 Step 2 — Install System Dependencies

In [ ]:
# ── 2.1  System packages (ffmpeg for audio, portaudio for sounddevice) ──────
# RunPod images already have most tools; this ensures ffmpeg is present.
!apt-get update -qq && apt-get install -y -qq \
    ffmpeg \
    libsndfile1 \
    portaudio19-dev \
    git \
    curl

print("System packages installed ✅")

---
## 🔧 Step 3 — Clone the Repository & Install Python Dependencies

In [ ]:
# ── 3.1  Clone the Moshi repository ────────────────────────────────────────
import os

WORKSPACE = "/workspace"           # RunPod persistent volume
REPO_DIR  = os.path.join(WORKSPACE, "moshi")

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/kyutai-labs/moshi.git {REPO_DIR}
    print("Repository cloned ✅")
else:
    print("Repository already exists, pulling latest changes…")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── 3.2  Ensure PyTorch is installed with CUDA support ──────────────────────
# If the base image already ships PyTorch + CUDA, this is a no-op.
# Uncomment and adjust the CUDA version (cu121 / cu118) to match your instance.

# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

import torch
assert torch.cuda.is_available(), (
    "PyTorch cannot see CUDA. "
    "Uncomment the pip install line above and re-run this cell."
)
print(f"PyTorch {torch.__version__} with CUDA {torch.version.cuda} ready ✅")

In [ ]:
# ── 3.3  Install the Moshi Python package  ───────────────────────────────────
# Option A  — install from PyPI (stable release)
!pip install -q -U moshi

# Option B  — install from the cloned repo (bleeding-edge)
# !pip install -q -e "{REPO_DIR}/moshi"

print("moshi package installed ✅")

In [ ]:
# ── 3.4  Install extra dependencies ─────────────────────────────────────────
!pip install -q \
    gradio \
    huggingface_hub \
    torchaudio \
    soundfile \
    sounddevice \
    sphn \
    sentencepiece \
    safetensors \
    einops \
    ipython \
    ipywidgets

print("Extra dependencies installed ✅")

In [ ]:
# ── 3.5  Verify all imports ─────────────────────────────────────────────────
import torch, torchaudio, soundfile, sphn, sentencepiece, safetensors
from huggingface_hub import hf_hub_download
from moshi.models import loaders, LMGen

print("All imports successful ✅")
print(f"  torch       {torch.__version__}")
print(f"  torchaudio  {torchaudio.__version__}")
print(f"  soundfile   {soundfile.__version__}")

---
## 🎯 Step 4 — Prepare Your Input `.wav` File

Moshi's audio codec **Mimi** requires:
- **Sample rate:** 24,000 Hz (24 kHz)
- **Channels:** Mono (1 channel)
- **Bit depth:** 16-bit PCM or float32

Run the cells below to upload or generate a sample file, then validate and convert it.

In [ ]:
# ── 4.1  Option A — Upload a file interactively (works in Jupyter) ─────────
# This will show a file-picker widget. Choose your .wav file.

from ipywidgets import FileUpload
import io, pathlib

uploader = FileUpload(accept='.wav,.mp3,.flac', multiple=False)
display(uploader)
print("👆 Click [Upload] and select your audio file.")
print("   Then run the next cell to save it.")

In [ ]:
# ── 4.1b  Save the uploaded file to disk ───────────────────────────────────
import os

AUDIO_DIR = "/workspace/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

if uploader.value:
    uploaded_file = list(uploader.value.values())[0]
    filename = list(uploader.value.keys())[0]
    input_path_raw = os.path.join(AUDIO_DIR, filename)
    with open(input_path_raw, "wb") as f:
        f.write(uploaded_file["content"])
    print(f"Saved uploaded file → {input_path_raw}")
else:
    print("No file uploaded yet. Run the cell above first, or use Option B below.")

In [ ]:
# ── 4.2  Option B — Download a sample wav file (if you don't have one) ──────
# A publicly available speech sample from Mozilla Common Voice / metavoice assets

import os

AUDIO_DIR = "/workspace/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

SAMPLE_URL  = "https://github.com/metavoiceio/metavoice-src/raw/main/assets/bria.mp3"
input_path_raw = os.path.join(AUDIO_DIR, "bria.mp3")

if not os.path.exists(input_path_raw):
    !wget -q -O {input_path_raw} {SAMPLE_URL}
    print(f"Sample audio downloaded → {input_path_raw}")
else:
    print(f"Sample audio already exists → {input_path_raw}")

In [ ]:
# ── 4.3  Validate & convert to 24 kHz mono WAV ──────────────────────────────
import torchaudio, soundfile as sf, os, torch

# 👇  Point this to whichever file you ended up with (uploaded or downloaded)
input_path_raw = "/workspace/audio/bria.mp3"   # ← change if you uploaded your own
input_path_wav = "/workspace/audio/input_24k.wav"

TARGET_SR = 24_000

# Load whatever format was provided
waveform, orig_sr = torchaudio.load(input_path_raw)
print(f"Original  : {waveform.shape}  |  {orig_sr} Hz  |  {waveform.shape[0]} channel(s)")

# Convert to mono if stereo
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)
    print("Converted to mono ✅")

# Resample to 24 kHz if needed
if orig_sr != TARGET_SR:
    resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=TARGET_SR)
    waveform  = resampler(waveform)
    print(f"Resampled {orig_sr} Hz → {TARGET_SR} Hz ✅")

# Save as 16-bit PCM WAV
torchaudio.save(input_path_wav, waveform, TARGET_SR, encoding="PCM_S", bits_per_sample=16)

duration = waveform.shape[-1] / TARGET_SR
print(f"\nReady input WAV   : {input_path_wav}")
print(f"  Shape    : {waveform.shape}")
print(f"  SR       : {TARGET_SR} Hz")
print(f"  Duration : {duration:.2f} s")

In [ ]:
# ── 4.4  Listen to the input audio inside the notebook ──────────────────────
from IPython.display import Audio, display
import soundfile as sf

data, sr = sf.read(input_path_wav)
print(f"Playing input  ({sr} Hz, {len(data)/sr:.2f} s)")
display(Audio(data, rate=sr))

---
## 🤖 Step 5 — Direct Python Inference (No Server Required)

This is the **recommended approach for batch processing** in a notebook.  
We load Mimi + Moshi directly, feed encoded audio tokens, and decode the output.

In [ ]:
# ── 5.1  Download & load model weights from Hugging Face ────────────────────
import torch
from huggingface_hub import hf_hub_download
from moshi.models import loaders, LMGen

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16          # use torch.float16 if your GPU does not support bfloat16

# Choose one of the two available Moshi voices:
#   kyutai/moshiko-pytorch-q8  (male  voice)
#   kyutai/moshika-pytorch-q8  (female voice)
HF_REPO = "kyutai/moshiko-pytorch-q8"

print(f"Device : {DEVICE}")
print(f"Dtype  : {DTYPE}")
print(f"Model  : {HF_REPO}")
print()

# Download checkpoint info (handles HF auth / caching automatically)
print("[1/4] Fetching checkpoint info…")
checkpoint_info = loaders.CheckpointInfo.from_hf_repo(HF_REPO)

print("[2/4] Loading Mimi (audio codec)…")
mimi = checkpoint_info.get_mimi(device=DEVICE)
mimi.set_num_codebooks(8)        # 8 codebooks for Moshi (max 32 supported by Mimi)
print("      Mimi loaded ✅")

print("[3/4] Loading text tokenizer…")
text_tokenizer = checkpoint_info.get_text_tokenizer()
print("      Text tokenizer loaded ✅")

print("[4/4] Loading Moshi LM (7B parameters — this may take a few minutes)…")
lm = checkpoint_info.get_moshi(device=DEVICE, dtype=DTYPE)
print("      Moshi LM loaded ✅")

print("\nAll models loaded! 🚀")

In [ ]:
# ── 5.2  Stream-encode input WAV, run Moshi, stream-decode output ────────────
import torch, sphn, time
from moshi.models import LMGen

# ── Load the prepared input WAV via sphn (same reader used internally) ───────
in_pcms, _ = sphn.read(input_path_wav, sample_rate=mimi.sample_rate)
in_pcms    = torch.from_numpy(in_pcms).to(device=DEVICE)   # [C, T]
in_pcms    = in_pcms[None, 0:1]                              # [B=1, C=1, T]

print(f"Input tensor  : {in_pcms.shape}  |  dtype={in_pcms.dtype}")

# ── Build the LM generator ───────────────────────────────────────────────────
lm_gen = LMGen(
    lm,
    temp=0.8,          # audio temperature  (higher → more varied)
    temp_text=0.7,     # text temperature
    use_sampling=True,
)

frame_size = int(mimi.sample_rate / mimi.frame_rate)   # 1920 samples @ 24kHz / 12.5Hz
print(f"Frame size    : {frame_size} samples  ({1000*frame_size/mimi.sample_rate:.1f} ms)")

out_wav_chunks = []
out_text_pieces = []

print("\nRunning streaming inference…")
t0 = time.time()

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    # Split input into fixed-size frames; Mimi v0.2.5+ requires complete frames only
    frames = in_pcms.split(frame_size, dim=-1)
    first  = True

    for step, frame in enumerate(frames):
        if frame.shape[-1] < frame_size:
            # Pad the last partial frame with silence
            pad_len = frame_size - frame.shape[-1]
            frame   = torch.nn.functional.pad(frame, (0, pad_len))

        codes = mimi.encode(frame)             # [B, K=8, 1]

        # First frame primes the transformer; output is None during the warmup delay
        if first:
            tokens_out = lm_gen.step(codes)
            if tokens_out is not None and max(lm_gen.lm_model.delays) == 0:
                wav_chunk  = mimi.decode(tokens_out[:, 1:])
                out_wav_chunks.append(wav_chunk.cpu())
            first = False
            continue

        tokens_out = lm_gen.step(codes)        # [B, 1+K, 1] or None during delay
        if tokens_out is None:
            continue

        # Decode audio from codebook tokens (skip token 0 which is the text token)
        wav_chunk  = mimi.decode(tokens_out[:, 1:])   # [B, 1, frame_size]
        out_wav_chunks.append(wav_chunk.cpu())

        # Optionally decode the text (inner monologue) token
        text_tok = tokens_out[0, 0].item()
        if text_tok not in (0, 3):   # skip padding / special tokens
            piece = text_tokenizer.id_to_piece(text_tok).replace("▁", " ")
            out_text_pieces.append(piece)
            print(piece, end="", flush=True)

        if step % 50 == 0:
            elapsed = time.time() - t0
            print(f"  [step {step:4d}  |  {elapsed:.1f}s elapsed]", flush=True)

dt = time.time() - t0
print(f"\n\nInference complete in {dt:.1f}s ✅")

In [ ]:
# ── 5.3  Concatenate chunks & save the output WAV ───────────────────────────
import torchaudio, os

OUTPUT_DIR  = "/workspace/audio"
output_path = os.path.join(OUTPUT_DIR, "moshi_output.wav")

if out_wav_chunks:
    out_wav = torch.cat(out_wav_chunks, dim=-1)   # [1, 1, T]
    out_wav = out_wav.squeeze(0)                   # [1, T]

    # Normalize to prevent clipping
    peak = out_wav.abs().max()
    if peak > 1.0:
        out_wav = out_wav / peak

    torchaudio.save(output_path, out_wav, mimi.sample_rate,
                    encoding="PCM_S", bits_per_sample=16)

    duration = out_wav.shape[-1] / mimi.sample_rate
    print(f"Output saved  : {output_path}")
    print(f"  Shape    : {out_wav.shape}")
    print(f"  SR       : {mimi.sample_rate} Hz")
    print(f"  Duration : {duration:.2f} s")
else:
    print("⚠️  No output chunks collected — check that the model received valid input frames.")

In [ ]:
# ── 5.4  Play the generated output audio in the notebook ─────────────────────
import soundfile as sf
from IPython.display import Audio, display

data_in,  sr_in  = sf.read(input_path_wav)
data_out, sr_out = sf.read(output_path)

print("▶  Input audio:")
display(Audio(data_in,  rate=sr_in))

print("▶  Moshi output audio:")
display(Audio(data_out, rate=sr_out))

if out_text_pieces:
    print("\n📝  Moshi inner monologue (text output):")
    print("".join(out_text_pieces))

In [ ]:
# ── 5.5  Optional: convert output to MP3 for easier sharing ─────────────────
output_mp3 = output_path.replace(".wav", ".mp3")
!ffmpeg -y -i {output_path} -codec:a libmp3lame -qscale:a 2 {output_mp3} -loglevel warning
print(f"MP3 saved: {output_mp3}")

---
## 🌐 Step 6 — Launch the Moshi Web Server (Interactive Mode)

The Moshi server exposes a WebSocket-based API and a built-in web UI.  
In RunPod, ports are accessible via the **HTTP Proxy** (port 8998) or a **Gradio public tunnel**.

### 6A — Via Gradio Public Tunnel (Simplest — works from anywhere)

In [ ]:
# ── 6A  Start Moshi server with Gradio tunnel ───────────────────────────────
# This creates a public *.gradio.live URL accessible from any browser.
# The tunnel goes through US servers — expect ~100–500ms extra latency.
#
# NOTE: This cell blocks until you interrupt the kernel (■ Stop button).

import subprocess, threading, time

cmd = [
    "python", "-m", "moshi.server",
    "--gradio-tunnel",
    "--hf-repo", "kyutai/moshiko-pytorch-q8",
    # Add --half for float16 if your GPU is pre-Ampere (no bfloat16 support)
    # "--half",
]

print("Starting Moshi server with Gradio tunnel…")
print("Watch for the line:  'Running on public URL: https://XXXX.gradio.live'")
print("Open that URL in any browser to interact with Moshi via the web UI.")
print("Press the STOP (■) button in the toolbar to shut down the server.\n")

!python -m moshi.server --gradio-tunnel --hf-repo kyutai/moshiko-pytorch-q8

### 6B — Via RunPod HTTP Port Proxy (No External Tunnel)

1. In the RunPod dashboard, open your pod → **Connect** → **HTTP Service**.
2. Add port **8998** to exposed ports.
3. RunPod will give you a URL like `https://<pod-id>-8998.proxy.runpod.net`.
4. Run the cell below (no `--gradio-tunnel`), then open that URL.

In [ ]:
# ── 6B  Start Moshi server on port 8998 for RunPod HTTP proxy ───────────────
# Access at: https://<pod-id>-8998.proxy.runpod.net
#
# NOTE: HTTP (not HTTPS) blocks microphone access in most browsers.
#       Use the SSL flags below or stick with Option 6A (Gradio tunnel).

import os

# Generate self-signed SSL certificates for HTTPS (needed for mic access)
SSL_DIR  = "/workspace/ssl"
KEY_PEM  = os.path.join(SSL_DIR, "key.pem")
CERT_PEM = os.path.join(SSL_DIR, "cert.pem")
os.makedirs(SSL_DIR, exist_ok=True)

if not os.path.exists(KEY_PEM):
    !openssl req -x509 -newkey rsa:4096 -keyout {KEY_PEM} -out {CERT_PEM} \
        -days 365 -nodes -subj "/CN=localhost" -quiet
    print(f"SSL certificates generated → {SSL_DIR}")
else:
    print(f"SSL certificates already exist → {SSL_DIR}")

print()
print("Starting Moshi server on port 8998 with HTTPS…")
print(f"Access via RunPod HTTP proxy:  https://<pod-id>-8998.proxy.runpod.net")
print("(Accept the self-signed cert warning in your browser.)\n")

!python -m moshi.server \
    --hf-repo kyutai/moshiko-pytorch-q8 \
    --ssl-keyfile  {KEY_PEM} \
    --ssl-certfile {CERT_PEM}

---
## 🔌 Step 7 — WebSocket API Client (Programmatic Access to a Running Server)

If you have the server running (from Step 6) in a **separate terminal / screen session**,  
you can interact with it programmatically using the built-in Python client.

In [ ]:
# ── 7.1  Launch the server in the background (screen / tmux alternative) ────
# This uses subprocess so the server runs concurrently with the rest of the notebook.
# The server log is written to /workspace/moshi_server.log

import subprocess, time, os

LOG_FILE = "/workspace/moshi_server.log"

server_proc = subprocess.Popen(
    [
        "python", "-m", "moshi.server",
        "--gradio-tunnel",
        "--hf-repo", "kyutai/moshiko-pytorch-q8",
    ],
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT,
)

print(f"Server PID : {server_proc.pid}")
print(f"Log file   : {LOG_FILE}")
print("Waiting for server to initialize (≈60–120 s for model loading)…")

# Poll the log file until the server announces it is ready
for _ in range(180):
    time.sleep(1)
    if os.path.exists(LOG_FILE):
        log = open(LOG_FILE).read()
        if "gradio.live" in log or "localhost:8998" in log:
            print("\nServer is ready! ✅")
            # Print the public URL
            for line in log.splitlines():
                if "gradio.live" in line or "Running on" in line:
                    print(" ", line.strip())
            break
else:
    print("Server did not report ready within 3 min — check the log file below.")
    print(open(LOG_FILE).read()[-3000:])

In [ ]:
# ── 7.2  Show live server log (re-run this cell to refresh) ─────────────────
import os
LOG_FILE = "/workspace/moshi_server.log"
if os.path.exists(LOG_FILE):
    print(open(LOG_FILE).read()[-5000:])
else:
    print("Log file not found — did you run cell 7.1?")

In [ ]:
# ── 7.3  Stop the background server ─────────────────────────────────────────
try:
    server_proc.terminate()
    server_proc.wait(timeout=10)
    print("Server stopped ✅")
except Exception as e:
    print(f"Could not stop server: {e}")

---
## 🔁 Step 8 — Batch Inference: Process Multiple WAV Files

In [ ]:
# ── 8.1  Batch processing using moshi.run_inference CLI ─────────────────────
# moshi.run_inference is the CLI-level wrapper that handles model loading,
# input reading, and output writing in a single command.

import os

INPUT_WAV  = "/workspace/audio/input_24k.wav"
OUTPUT_WAV = "/workspace/audio/batch_output.wav"

#  --batch-size 1 is safest; increase only if you replicate the same audio N times
#  (the model requires all batch items to have identical frame counts)
!python -m moshi.run_inference \
    --hf-repo kyutai/moshiko-pytorch-q8 \
    --device cuda \
    --batch-size 1 \
    {INPUT_WAV} \
    {OUTPUT_WAV}

print(f"\nOutput written to: {OUTPUT_WAV}")

In [ ]:
# ── 8.2  Play batch inference result ────────────────────────────────────────
import soundfile as sf
from IPython.display import Audio, display

if os.path.exists(OUTPUT_WAV):
    data, sr = sf.read(OUTPUT_WAV)
    duration  = len(data) / sr
    print(f"▶  Batch output  ({sr} Hz, {duration:.2f} s)")
    display(Audio(data, rate=sr))
else:
    print("Output file not found — check the cell above for errors.")

---
## 🧠 Step 9 — Mimi-Only Encode / Decode (Audio Codec Demo)

You can use **Mimi** independently for audio compression / reconstruction,  
without running the full 7B Moshi language model.

In [ ]:
# ── 9.1  Mimi encode → decode round-trip ────────────────────────────────────
import torch, torchaudio, soundfile as sf
from huggingface_hub import hf_hub_download
from moshi.models import loaders
from IPython.display import Audio, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Mimi codec…")
mimi_weight = hf_hub_download(loaders.DEFAULT_REPO, loaders.MIMI_NAME)
mimi = loaders.get_mimi(mimi_weight, device=DEVICE)
mimi.set_num_codebooks(8)
print("Mimi loaded ✅")

# Load input
wav, sr = torchaudio.load("/workspace/audio/input_24k.wav")
wav = wav.unsqueeze(0).to(DEVICE)   # [1, 1, T]

print(f"Input shape : {wav.shape}")

# Encode → decode (non-streaming, full-sequence)
with torch.no_grad():
    codes   = mimi.encode(wav)       # [1, 8, T/frame_rate]
    decoded = mimi.decode(codes)     # [1, 1, T]

print(f"Codes shape : {codes.shape}")
print(f"  Codebooks : {codes.shape[1]}")
print(f"  Frames    : {codes.shape[2]}  ({codes.shape[2]/mimi.frame_rate:.2f} s)")

# Save reconstructed audio
recon_path = "/workspace/audio/mimi_reconstructed.wav"
torchaudio.save(recon_path, decoded.squeeze(0).cpu(), mimi.sample_rate)
print(f"\nReconstructed audio → {recon_path}")

data_rec, sr_rec = sf.read(recon_path)
print("▶  Mimi reconstructed audio (encode → decode):")
display(Audio(data_rec, rate=sr_rec))

In [ ]:
# ── 10.1  Fix: Set HuggingFace token (if models are gated / rate-limited) ───
import os

# Paste your token from https://huggingface.co/settings/tokens
HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # ← replace this

if HF_TOKEN != "hf_YOUR_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    !huggingface-cli login --token {HF_TOKEN} --add-to-git-credential
    print("HuggingFace token set ✅")
else:
    print("⚠️  Replace HF_TOKEN with your actual token from huggingface.co/settings/tokens")

In [ ]:
# ── 10.2  Fix: CUDA Out-of-Memory — free GPU cache & reduce precision ────────
import torch, gc

print("Before cleanup:")
print(f"  Allocated : {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
print(f"  Reserved  : {torch.cuda.memory_reserved()  / 1024**2:.1f} MB")

gc.collect()
torch.cuda.empty_cache()

print("\nAfter cleanup:")
print(f"  Allocated : {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
print(f"  Reserved  : {torch.cuda.memory_reserved()  / 1024**2:.1f} MB")

print("""
Tips for memory-constrained GPUs:
  • Use --half flag  → float16 instead of bfloat16 (~same VRAM, older GPU support)
  • Use Q8 model    → kyutai/moshiko-pytorch-q8  (requires Rust backend)
  • Use batch_size=1 (default for run_inference)
  • See: https://github.com/kyutai-labs/moshi/issues/54  (12 GB GPU guide)
""")

In [ ]:
# ── 10.3  Fix: Partial frame assertion error ─────────────────────────────────
# Since Moshi v0.2.5a, Mimi does NOT accept partial frames in streaming mode.
# Always pad your input to the nearest multiple of frame_size (1920 samples).

import torch

def pad_to_frame_multiple(wav: torch.Tensor, frame_size: int = 1920) -> torch.Tensor:
    """Pad `wav` (shape [B, 1, T]) to the next multiple of `frame_size`."""
    T = wav.shape[-1]
    remainder = T % frame_size
    if remainder != 0:
        pad_len = frame_size - remainder
        wav = torch.nn.functional.pad(wav, (0, pad_len))
    return wav

# Example usage
dummy = torch.randn(1, 1, 50000)
padded = pad_to_frame_multiple(dummy)
print(f"Original shape : {dummy.shape}")
print(f"Padded shape   : {padded.shape}  (divisible by 1920: {padded.shape[-1] % 1920 == 0})")

In [ ]:
# ── 10.4  Fix: Check if port 8998 is accessible inside the pod ──────────────
import socket

def is_port_open(host: str, port: int, timeout: float = 2.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except (OSError, ConnectionRefusedError):
        return False

port = 8998
status = is_port_open("localhost", port)
print(f"Port {port} (localhost): {'OPEN ✅' if status else 'CLOSED ❌ — server may not be running'}")

# Get the RunPod pod's public hostname / IP
import os
pod_id = os.environ.get("RUNPOD_POD_ID", "<unknown>")
print(f"\nRunPod pod ID : {pod_id}")
print(f"HTTP proxy URL: https://{pod_id}-{port}.proxy.runpod.net")
print("  (Ensure port 8998 is enabled in your pod's HTTP ports settings)")

In [ ]:
# ── 10.5  Fix: Disable torch.compile for Python 3.14+ ──────────────────────
import sys, os

if sys.version_info >= (3, 14):
    os.environ["NO_TORCH_COMPILE"] = "1"
    print("NO_TORCH_COMPILE=1 set for Python 3.14+ ✅")
else:
    print(f"Python {sys.version_info.major}.{sys.version_info.minor} — torch.compile supported, no fix needed.")

---
## 🧹 Step 11 — Cleanup & Resource Management

In [ ]:
# ── 11.1  Free GPU memory after inference session ───────────────────────────
import gc, torch

# Delete model references
for obj_name in ("lm", "mimi", "lm_gen", "checkpoint_info"):
    if obj_name in dir():
        del globals()[obj_name]

gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after cleanup:")
print(f"  Allocated : {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
print(f"  Reserved  : {torch.cuda.memory_reserved()  / 1024**2:.1f} MB")

In [ ]:
# ── 11.2  List all generated audio files ────────────────────────────────────
import os

AUDIO_DIR = "/workspace/audio"
print(f"Files in {AUDIO_DIR}:\n")
for f in sorted(os.listdir(AUDIO_DIR)):
    path  = os.path.join(AUDIO_DIR, f)
    size  = os.path.getsize(path)
    print(f"  {f:<35}  {size/1024:>8.1f} KB")